# Hybrid GraphRAG Pipeline - Full SOTA Implementation

This notebook demonstrates the full hybrid pipeline combining:
- BM25 keyword retrieval
- Dense retrieval (BGE-M3 + FAISS)
- Citation graph retrieval (PPR ranking)
- Weighted Reciprocal Rank Fusion (RRF)
- Cross-encoder reranking (BGE-reranker-v2-m3)
- LLM verification (Qwen2.5-7B GGUF)

## Pipeline Components
1. **Retrieval Stage**: BM25 + Dense + Graph → multiple signal sources
2. **Fusion Stage**: Combine signals using Weighted RRF
3. **Reranking Stage**: Re-rank fused results with cross-encoder
4. **Verification Stage**: Filter results with LLM verifier
5. **Normalization**: Canonicalize all citations

## Experiment Presets
- `exp_baseline`: BM25 only
- `exp_dense_only`: Dense retrieval only
- `exp_bm25_dense`: BM25 + Dense fusion
- `exp_full_retrieval`: BM25 + Dense + Graph
- `exp_full_rrf`: Full retrieval + RRF fusion
- `exp_full_reranker`: Full retrieval + RRF + Reranker
- `exp_full_pipeline`: Full pipeline with all components

In [ ]:
# Helper functions for uv cache archive management
import hashlib
import subprocess
from pathlib import Path


def compute_cache_hash(cache_dir):
    """Compute MD5 hash of all files in cache directory."""
    md5 = hashlib.md5()
    if not os.path.exists(cache_dir):
        return None

    # Get all files, sort for consistency
    files = sorted(Path(cache_dir).rglob("*"), key=lambda p: str(p))
    for f in files:
        if f.is_file():
            # Hash relative path and file content
            rel_path = str(f.relative_to(cache_dir)).encode()
            md5.update(rel_path)
            try:
                with open(f, "rb") as fp:
                    # Read in chunks to handle large files
                    for chunk in iter(lambda: fp.read(8192), b""):
                        md5.update(chunk)
            except OSError:
                pass  # Skip files we can't read
    return md5.hexdigest()


def create_uv_cache_archive(cache_dir, output_path, hash_path=None):
    """Create tar.gz archive of uv cache and optional hash file."""
    try:
        # Check available space (need ~2x archive size for safety)
        stat = os.statvfs("/tmp")
        free_space = stat.f_bavail * stat.f_frsize / 1e9
        if free_space < 5:
            print(f"  ⚠ Warning: Low disk space ({free_space:.1f}GB free)")

        # Create archive
        print(f"Creating archive from {cache_dir}...")
        result = subprocess.run(
            ["tar", "-czf", str(output_path), "-C", "/tmp", "uv-cache"],
            capture_output=True,
            text=True,
        )

        if result.returncode != 0:
            print(f"  ✗ Archive creation failed: {result.stderr}")
            return False

        # Verify archive
        verify = subprocess.run(["gunzip", "-t", str(output_path)], capture_output=True, text=True)

        if verify.returncode != 0:
            print("  ✗ Archive verification failed")
            return False

        archive_size = os.path.getsize(output_path) / 1e9
        print(f"  ✓ Archive created: {archive_size:.2f}GB")

        # Create hash file if requested
        if hash_path:
            cache_hash = compute_cache_hash(cache_dir)
            if cache_hash:
                with open(hash_path, "w") as f:
                    f.write(cache_hash)
                print(f"  ✓ Hash saved: {cache_hash[:16]}...")

        return True
    except Exception as e:
        print(f"  ✗ Error creating archive: {e}")
        return False


def copy_archive_to_drive(local_archive, drive_archive, local_hash=None, drive_hash=None):
    """Copy archive and optional hash to Drive."""
    try:
        # Copy archive
        result = subprocess.run(
            ["cp", local_archive, drive_archive], capture_output=True, text=True
        )
        if result.returncode != 0:
            print(f"  ✗ Failed to copy archive to Drive: {result.stderr}")
            return False

        print("  ✓ Archive copied to Drive")

        # Copy hash if provided
        if local_hash and drive_hash:
            result = subprocess.run(["cp", local_hash, drive_hash], capture_output=True, text=True)
            if result.returncode == 0:
                print("  ✓ Hash copied to Drive")

        return True
    except Exception as e:
        print(f"  ✗ Error copying to Drive: {e}")
        return False


print("✓ Archive helper functions loaded")

## 1. Colab Setup (Standalone)

This section handles Drive mount, cache setup, and repo configuration.
Works standalone - no need to run colab_startup.ipynb first.

In [ ]:
# @title Repository Configuration

repo_url = "https://github.com/samin-irtiza/Omnilex-Agentic-Retrieval-Competition.git"  # @param {type:"string"}  # noqa: E501
branch_name = "hybrid-graphrag"  # @param {type:"string"}

repo_dir = "/content/Omnilex-Agentic-Retrieval-Competition"

print(f"Repository: {repo_url}")
print(f"Branch: {branch_name}")
print(f"Local directory: {repo_dir}")

In [ ]:
# Step 1: Mount Google Drive (if in Colab)
import os

try:
    from google.colab import drive

    COLAB_ENV = True
    print("Mounting Google Drive...")
    drive.mount("/content/drive")

    # Verify mount
    if os.path.exists("/content/drive/MyDrive"):
        print("✓ Google Drive mounted successfully")
    else:
        print("✗ Drive mount failed - please check authentication")
        raise RuntimeError("Drive mount failed")
except ImportError:
    COLAB_ENV = False
    print("Local environment - skipping Drive mount")

In [ ]:
# Step 2: Configure uv cache (use local dir, synced with Drive)
import os
import shutil

drive_cache_dir = "/content/drive/MyDrive/omnilex-cache"
local_cache_dir = "/tmp/uv-cache"
archive_name = "uv-cache.tar.gz"
hash_name = "uv-cache.hash"

# Use local cache for uv (Drive doesn't support file locking)
os.environ["UV_CACHE_DIR"] = local_cache_dir
os.makedirs(local_cache_dir, exist_ok=True)

print(f"✓ UV_CACHE_DIR set to local path: {local_cache_dir}")

# Sync cached wheels from Drive to local cache using archive
if COLAB_ENV:
    drive_archive = os.path.join(drive_cache_dir, archive_name)
    local_archive = os.path.join("/tmp", archive_name)

    if os.path.exists(drive_archive):
        print(f"Found archive on Drive: {archive_name}")
        print("Copying archive from Drive (fast single-file read)...")

        # Copy archive from Drive to /tmp
        result = subprocess.run(
            ["cp", drive_archive, local_archive], capture_output=True, text=True
        )

        if result.returncode == 0:
            print("  ✓ Archive copied to /tmp")

            # Verify archive integrity
            verify = subprocess.run(["gunzip", "-t", local_archive], capture_output=True, text=True)

            if verify.returncode == 0:
                print("  ✓ Archive verification passed")

                # Extract archive to local cache
                print(f"Extracting archive to {local_cache_dir}...")
                extract = subprocess.run(
                    ["tar", "-xzf", local_archive, "-C", "/tmp/"], capture_output=True, text=True
                )

                if extract.returncode == 0:
                    print("  ✓ Cache extracted from archive")
                    # Clean up archive
                    os.remove(local_archive)
                else:
                    print(f"  ✗ Archive extraction failed: {extract.stderr}")
                    print("  Falling back to PyPI downloads")
            else:
                print("  ⚠ Archive verification failed - may be corrupted")
                print("  Falling back to PyPI downloads")
                os.remove(local_archive) if os.path.exists(local_archive) else None
        else:
            print(f"  ✗ Failed to copy archive: {result.stderr}")
            print("  Falling back to PyPI downloads")
    else:
        print(f"⚠ No archive found on Drive: {drive_archive}")
        print("  Will download packages from PyPI")
        print("  Tip: After first install, run the manual archive update cell")
else:
    print("Local environment - skipping Drive sync")

# Display local cache size
if Path(local_cache_dir).exists():
    cache_size = (
        sum(f.stat().st_size for f in Path(local_cache_dir).rglob("*") if f.is_file()) / 1e6
    )
    print(f"Local cache size: {cache_size:.1f} MB")
else:
    print("Local cache empty")

In [ ]:
# Step 3: Clone or update repository
import os
import subprocess

if not os.path.exists(repo_dir):
    print(f"Cloning repository to {repo_dir}...")
    result = subprocess.run(
        ["git", "clone", "-b", branch_name, repo_url, repo_dir], capture_output=True, text=True
    )
    if result.returncode == 0:
        print(f"✓ Repository cloned successfully (branch: {branch_name})")
    else:
        print(f"✗ Clone failed: {result.stderr}")
        raise RuntimeError("Repository clone failed")
else:
    print(f"Repository exists, pulling updates (branch: {branch_name})...")
    os.chdir(repo_dir)
    result = subprocess.run(["git", "pull", "origin", branch_name], capture_output=True, text=True)
    if result.returncode == 0:
        print("✓ Repository updated successfully")
    else:
        print(f"✗ Pull failed: {result.stderr}")
        print("  Continuing with existing code...")

# Change to repo directory
os.chdir(repo_dir)
print(f"✓ Working directory: {os.getcwd()}")

In [ ]:
# Step 4: Symlink data/ from Drive cache
import os

drive_data_cache = "/content/drive/MyDrive/omnilex-cache/data"
local_data_dir = os.path.join(repo_dir, "data")

# Check if Drive cache exists and has content
if COLAB_ENV and os.path.exists(drive_data_cache) and os.listdir(drive_data_cache):
    # Remove existing data directory if it exists
    if os.path.exists(local_data_dir):
        if os.path.islink(local_data_dir):
            os.unlink(local_data_dir)
        else:
            import shutil

            shutil.rmtree(local_data_dir)

    # Create symlink
    os.symlink(drive_data_cache, local_data_dir)
    print(f"✓ Symlinked {local_data_dir} -> {drive_data_cache}")
else:
    if COLAB_ENV:
        print(f"⚠ Drive data cache empty or missing: {drive_data_cache}")
        print("  Skipping data symlink - will need to download data manually")
    else:
        print("Local environment - using local data directory")

In [ ]:
# Step 5: Symlink data/processed/ (indices) from Drive cache
import os

drive_indices_cache = "/content/drive/MyDrive/omnilex-cache/indices"
local_processed_dir = os.path.join(repo_dir, "data", "processed")

# Check if Drive cache exists and has content
if COLAB_ENV and os.path.exists(drive_indices_cache) and os.listdir(drive_indices_cache):
    # Ensure parent directory exists
    os.makedirs(os.path.join(repo_dir, "data"), exist_ok=True)

    # Remove existing processed directory if it exists
    if os.path.exists(local_processed_dir):
        if os.path.islink(local_processed_dir):
            os.unlink(local_processed_dir)
        else:
            import shutil

            shutil.rmtree(local_processed_dir)

    # Create symlink
    os.symlink(drive_indices_cache, local_processed_dir)
    print(f"✓ Symlinked {local_processed_dir} -> {drive_indices_cache}")
else:
    if COLAB_ENV:
        print(f"⚠ Drive indices cache empty or missing: {drive_indices_cache}")
        print("  Skipping indices symlink - will need to rebuild indices")
    else:
        print("Local environment - using local indices directory")

In [ ]:
# Step 6: Install dependencies using uv with local cache + auto-update archive
import os
import subprocess
from pathlib import Path

print("Installing dependencies with uv (using cache)...")
print(f"UV_CACHE_DIR: {os.environ.get('UV_CACHE_DIR')}")

result = subprocess.run(["uv", "pip", "install", "-e", ".[dev]"], capture_output=True, text=True)

if result.returncode == 0:
    print("✓ Dependencies installed successfully")
else:
    print(f"✗ Installation failed: {result.stderr}")
    # Try without -e flag as fallback
    print("\nTrying without editable install...")
    result = subprocess.run(["uv", "pip", "install", "."], capture_output=True, text=True)
    if result.returncode == 0:
        print("✓ Dependencies installed successfully (non-editable)")
    else:
        print(f"✗ Installation failed: {result.stderr}")

# Auto-update archive if cache changed (only in Colab)
if COLAB_ENV:
    print("\nChecking if uv cache changed...")
    local_cache_dir = "/tmp/uv-cache"
    drive_cache_dir = "/content/drive/MyDrive/omnilex-cache"
    archive_name = "uv-cache.tar.gz"
    hash_name = "uv-cache.hash"

    new_hash = compute_cache_hash(local_cache_dir)

    if new_hash:
        drive_hash_path = os.path.join(drive_cache_dir, hash_name)

        # Read old hash from Drive
        old_hash = None
        if os.path.exists(drive_hash_path):
            with open(drive_hash_path) as f:
                old_hash = f.read().strip()

        if old_hash != new_hash:
            print("Cache changed detected!")
            print(f"  Old hash: {old_hash[:16] if old_hash else 'None'}...")
            print(f"  New hash: {new_hash[:16]}...")

            # Create new archive
            local_archive = os.path.join("/tmp", archive_name)
            local_hash = os.path.join("/tmp", hash_name)

            if create_uv_cache_archive(local_cache_dir, local_archive, local_hash):
                # Copy to Drive
                drive_archive = os.path.join(drive_cache_dir, archive_name)
                drive_hash_file = os.path.join(drive_cache_dir, hash_name)

                if copy_archive_to_drive(local_archive, drive_archive, local_hash, drive_hash_file):
                    print("✓ Archive updated on Drive")

                # Cleanup local files
                os.remove(local_archive) if os.path.exists(local_archive) else None
                os.remove(local_hash) if os.path.exists(local_hash) else None
        else:
            print("✓ Cache unchanged, skipping archive update")
    else:
        print("⚠ Could not compute cache hash")

In [ ]:
# Manual Archive Update Cell (run this to force-update the archive on Drive)
import os
import subprocess

if COLAB_ENV:
    drive_cache_dir = "/content/drive/MyDrive/omnilex-cache"
    local_cache_dir = "/tmp/uv-cache"
    archive_name = "uv-cache.tar.gz"
    hash_name = "uv-cache.hash"

    print("=== Manual Archive Update ===")

    # Check if local cache exists
    if not os.path.exists(local_cache_dir):
        print("✗ Local cache not found. Run Step 2 first.")
    else:
        # Create archive
        local_archive = os.path.join("/tmp", archive_name)
        local_hash = os.path.join("/tmp", hash_name)

        if create_uv_cache_archive(local_cache_dir, local_archive, local_hash):
            # Verify archive
            verify = subprocess.run(["tar", "-tzf", local_archive], capture_output=True, text=True)

            if verify.returncode == 0:
                file_count = len(verify.stdout.strip().split("\n"))
                print(f"  ✓ Archive verified ({file_count} files)")

                # Copy to Drive
                drive_archive = os.path.join(drive_cache_dir, archive_name)
                drive_hash_file = os.path.join(drive_cache_dir, hash_name)

                if copy_archive_to_drive(local_archive, drive_archive, local_hash, drive_hash_file):
                    print("✓ Archive manually updated on Drive")
            else:
                print("  ✗ Archive verification failed")

            # Cleanup
            os.remove(local_archive) if os.path.exists(local_archive) else None
            os.remove(local_hash) if os.path.exists(local_hash) else None
        else:
            print("✗ Failed to create archive")
else:
    print("This cell is for Colab environment only.")

In [ ]:
# Step 7: Verification
import os
import subprocess
from pathlib import Path

print("=== Setup Verification ===")

# Check Drive mount
if COLAB_ENV:
    drive_mounted = os.path.exists("/content/drive/MyDrive")
    print(f"Drive mounted: {'✓' if drive_mounted else '✗'}")

# Check repo
repo_exists = os.path.exists(repo_dir)
print(f"Repository exists: {'✓' if repo_exists else '✗'}")

# Check uv cache (local)
uv_cache = os.environ.get("UV_CACHE_DIR", "not set")
print(f"UV_CACHE_DIR (local): {uv_cache}")
if os.path.exists(uv_cache):
    cache_size = sum(f.stat().st_size for f in Path(uv_cache).rglob("*") if f.is_file()) / 1e6
    print(f"  Cache size: {cache_size:.1f} MB")

# Check Drive archive (new)
if COLAB_ENV:
    drive_cache_dir = "/content/drive/MyDrive/omnilex-cache"
    archive_path = os.path.join(drive_cache_dir, "uv-cache.tar.gz")
    hash_path = os.path.join(drive_cache_dir, "uv-cache.hash")

    print(f"\nDrive archive exists: {'✓' if os.path.exists(archive_path) else '✗'}")
    if os.path.exists(archive_path):
        archive_size = os.path.getsize(archive_path) / 1e9
        print(f"  Archive size: {archive_size:.2f} GB")

        # Verify archive integrity
        verify = subprocess.run(["gunzip", "-t", archive_path], capture_output=True, text=True)
        print(f"  Archive valid: {'✓' if verify.returncode == 0 else '✗'}")

    print(f"Drive hash exists: {'✓' if os.path.exists(hash_path) else '✗'}")
    if os.path.exists(hash_path):
        with open(hash_path) as f:
            print(f"  Hash: {f.read().strip()[:16]}...")

# Check data symlink
data_dir = os.path.join(repo_dir, "data")
data_linked = os.path.islink(data_dir) if os.path.exists(data_dir) else False
print(f"\nData symlinked: {'✓' if data_linked else '✗'}")
if data_linked:
    print(f"  -> {os.readlink(data_dir)}")

# Check indices symlink
processed_dir = os.path.join(repo_dir, "data", "processed")
indices_linked = os.path.islink(processed_dir) if os.path.exists(processed_dir) else False
print(f"Indices symlinked: {'✓' if indices_linked else '✗'}")
if indices_linked:
    print(f"  -> {os.readlink(processed_dir)}")

print("\n=== Setup Complete ===")

## 2. Configuration

In [ ]:
import os
import sys
from pathlib import Path

# Configuration
DATASET_MODE = "val"  # Change to "test" for final submission
REPO_ROOT = Path(repo_dir)
DATA_PATH = REPO_ROOT / "data" / "raw"
OUTPUT_PATH = REPO_ROOT / "data" / "processed"
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

# Add the project's 'src' directory to sys.path to make modules discoverable.
src_dir_path = REPO_ROOT / "src"
if src_dir_path.exists() and str(src_dir_path.resolve()) not in sys.path:
    sys.path.insert(0, str(src_dir_path.resolve()))

from omnilex.retrieval.ablation.config import EXPERIMENT_PRESETS, ExperimentConfig
from omnilex.retrieval.ablation.runner import ExperimentRunner

print(f"Environment: {'Colab' if COLAB_ENV else 'Local'}")
print(f"Repo root: {REPO_ROOT}")
print(f"Data path: {DATA_PATH}")
print(f"Output path: {OUTPUT_PATH}")
print(f"Available presets: {list(EXPERIMENT_PRESETS.keys())}")

## 3. Load Queries

In [ ]:
import pandas as pd

QUERY_FILE = DATA_PATH / f"{DATASET_MODE}.csv"
if not QUERY_FILE.exists():
    raise FileNotFoundError(f"Query file not found: {QUERY_FILE}")

test_df = pd.read_csv(QUERY_FILE)
print(f"Loaded {len(test_df)} queries from {QUERY_FILE}")

# Format queries for runner
formatted_queries = [
    {"id": row["query_id"], "query": row["query"]} for _, row in test_df.iterrows()
]

# Load ground truth if available
ground_truth = None
if "gold_citations" in test_df.columns:
    ground_truth = {}
    for _, row in test_df.iterrows():
        citations = [c.strip() for c in str(row["gold_citations"]).split(";") if c.strip()]
        ground_truth[row["query_id"]] = citations
    print(f"Loaded ground truth for {len(ground_truth)} queries")

test_df.head()

## 3. Run Full Pipeline Experiment

In [ ]:
# Run the full pipeline experiment
config = ExperimentConfig.from_preset("exp_baseline")
config.laws_corpus_path = DATA_PATH / "laws_de.csv"
config.index_cache_dir = OUTPUT_PATH  # Optional

runner = ExperimentRunner(config=config, output_dir=OUTPUT_PATH / "experiments")

print(f"Running experiment: {config.name}")
print(f"Components enabled: {config.components}")

results = runner.run(queries=formatted_queries, ground_truth=ground_truth)

print("\nExperiment complete!")
print(f"Aggregate metrics: {results['metrics']}")

## 4. Compare Different Presets

In [ ]:
# Compare multiple presets
presets_to_compare = ["exp_baseline", "exp_bm25_dense", "exp_full_retrieval", "exp_full_pipeline"]

comparison_results = []

for preset_name in presets_to_compare:
    print(f"\n{'=' * 50}")
    print(f"Running preset: {preset_name}")

    config = ExperimentConfig.from_preset(preset_name)
    config.name = preset_name

    runner = ExperimentRunner(config=config, output_dir=OUTPUT_PATH / "experiments" / preset_name)

    results = runner.run(formatted_queries, ground_truth)
    metrics = results["metrics"]

    comparison_results.append(
        {
            "preset": preset_name,
            "macro_f1": metrics.get("macro_f1", 0),
            "macro_precision": metrics.get("macro_precision", 0),
            "macro_recall": metrics.get("macro_recall", 0),
        }
    )

    print(f"Macro F1: {metrics.get('macro_f1', 0):.4f}")

# Show comparison table
comparison_df = pd.DataFrame(comparison_results)
print(f"\n{'=' * 50}")
print("PRESET COMPARISON")
print(f"{'=' * 50}")
print(comparison_df.to_string(index=False))

## 5. Generate Submission

In [ ]:
import json

# Generate final submission from best experiment
submission_path = OUTPUT_PATH / "submission.csv"

# Load the best results (from full pipeline)
best_results_path = (
    OUTPUT_PATH
    / "experiments"
    / "full_pipeline_experiment"
    / "full_pipeline_experiment_results.json"
)

if best_results_path.exists():
    with open(best_results_path) as f:
        best_results = json.load(f)

    # Convert to submission format
    submission_df = pd.DataFrame(best_results["results"])
    submission_df["predicted_citations"] = submission_df["citations"].apply(lambda x: ";".join(x))
    submission_df[["query_id", "predicted_citations"]].to_csv(submission_path, index=False)

    print(f"Submission saved to: {submission_path}")
    print(f"Total queries: {len(submission_df)}")
    print("\nSample submission:")
    print(submission_df[["query_id", "predicted_citations"]].head())
else:
    print(f"Best results file not found: {best_results_path}")

## 6. Evaluate Results

In [ ]:
# Run validation if ground truth exists
if ground_truth:
    print("Running evaluation...")
    import subprocess

    result = subprocess.run(
        ["python", "scripts/evaluate_submission.py", str(submission_path)],
        capture_output=True,
        text=True,
    )
    print(result.stdout)
    if result.stderr:
        print(f"Errors: {result.stderr}")
else:
    print("No ground truth available, skipping evaluation")